<a href="https://colab.research.google.com/github/Andrea31-21/Curso2026-2027/blob/master/Assignment4/course_materials/notebooks/Task04_puzzle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Code to order**

# **Task 04: Graph querying**

In [12]:
!pip install rdflib
github_storage = "https://raw.githubusercontent.com/FacultadInformatica-LinkedData/Curso2026-2027/refs/heads/master/Assignment4/course_materials"

In [13]:
from rdflib import Graph, Namespace, Literal
g = Graph()
g.parse(github_storage+"/rdf/example3.rdf", format="xml")

from rdflib.plugins.sparql import prepareQuery
from rdflib import XSD
from rdflib import FOAF

VCARD = Namespace("http://www.w3.org/2001/vcard-rdf/3.0#")

Spanish: Listar todos los recursos que contienen la propiedad VCARD:FN

English: List all resources that contain the VCARD:FN property

In [14]:
q = prepareQuery('''
  SELECT ?Subject WHERE {
    ?Subject vcard:FN ?FullName.
  }
  ''',
  initNs = { "vcard": VCARD}
)

for r in g.query(q):
  print(r.Subject)

http://somewhere#JaneSmith
http://somewhere#JohnSmith
http://somewhere#SarahJones
http://somewhere#MattJones


Spanish Repetir la anterior consulta, pero pidiendo ahora además los nombres completos de los sujetos

English: Repeat the previous query, but this time also request the full names of the subjects

In [15]:
q = prepareQuery('''
  SELECT ?Subject ?FullName WHERE {
    ?Subject vcard:FN ?FullName.
  }
  ''',
  initNs = { "vcard": VCARD}
)

for r in g.query(q):
  print(r.Subject, r.FullName)

http://somewhere#JaneSmith Jane Smith
http://somewhere#JohnSmith John Smith
http://somewhere#SarahJones Sarah Jones
http://somewhere#MattJones Matt Jones


Spanish: Obtener todos los recursos que contienen "Smith" como nombre de familia

English: Retrieve all resources containing ‘Smith’ as a surname

In [16]:
q = prepareQuery('''
  SELECT ?Subject WHERE {
    ?Subject vcard:Family ?Family.
  }
  ''',
  initNs = { "vcard": VCARD}
)

for r in g.query(q, initBindings = {'?Family' : Literal('Smith', datatype=XSD.string)}):
  print(r.Subject)

http://somewhere#JaneSmith
http://somewhere#JohnSmith


Spanish: Obtener todos los elementos que contienen un email asociado

English: Retrieve all items that have an associated email address



In [17]:
q = prepareQuery('''
  SELECT ?Subject ?Email WHERE {
    ?Subject foaf:email ?Email.
  }
  ''',
  initNs = { "foaf": FOAF}
)

for r in g.query(q):
  print(r.Subject,r.Email)

http://somewhere#JaneSmith jSmith@somewhere.com
http://somewhere#SarahJones sJones@somewhere.com


Spanish: Consultar todos los que conocen (FOAF:knows) a "Jane Smith" y obtenemos sus nombres de pila (VCARD:Given)

English: Query everyone who knows (FOAF:knows) “Jane Smith” and retrieve their first names (VCARD:Given)

In [18]:
q = prepareQuery('''
  SELECT  ?Subject ?Given  WHERE {
    ?Subject foaf:knows ?JaneSmith.
	?JaneSmith vcard:FN ?JaneSmithFullName.
	?Subject vcard:Given ?Given.
  }
  ''',
  initNs = { "foaf": FOAF, "vcard": VCARD, "xsd":XSD}
)

for r in g.query(q, initBindings = {'?JaneSmithFullName' : Literal('Jane Smith', datatype=XSD.string)}):
  print(r.Subject, r.Given)

http://somewhere#JohnSmith John
http://somewhere#MattJones Matt


# **Task 04: Do the following exercises**

Spanish: Listar el nombre completo y el email de quienes tengan ambos

English: List the full name and email of those who have both

In [19]:
q = prepareQuery('''
  SELECT ?Subject ?FullName ?Email WHERE {
    ?Subject vcard:FN ?FullName.
    ?Subject foaf:email ?Email.
  }
''',
  initNs = { "vcard": VCARD, "foaf": FOAF }
)

for r in g.query(q):
  print(r.Subject, r.FullName, r.Email)

http://somewhere#JaneSmith Jane Smith jSmith@somewhere.com
http://somewhere#SarahJones Sarah Jones sJones@somewhere.com


Spanish: Listar todos los nombres completos ordenados alfabéticamente (ORDER BY)

English: List all full names in alphabetical order (ORDER BY)

In [20]:
q = prepareQuery('''
  SELECT ?FullName WHERE {
    ?Subject vcard:FN ?FullName.
  }
  ORDER BY ?FullName
''',
  initNs = { "vcard": VCARD }
)

for r in g.query(q):
  print(r.FullName)

Jane Smith
John Smith
Matt Jones
Sarah Jones


Spanish: Listar el nombre completo de quienes no tienen email

English: List the fullname of those who do not have an email

In [21]:
q = prepareQuery('''
  SELECT ?FullName WHERE {
    ?Subject vcard:FN ?FullName.
    FILTER NOT EXISTS {
      ?Subject foaf:email ?Email.
    }
  }
''',
  initNs = { "vcard": VCARD, "foaf": FOAF }
)

for r in g.query(q):
  print(r.FullName)

John Smith
Matt Jones


Spanish: Comprobar si existe algún recurso con el apellido "García"

English: Check whether any resource has the surname "García"

In [22]:
q = prepareQuery('''
  ASK {
    ?Subject vcard:Family ?Family.
    FILTER(?Family = "García" || ?Family = "Garcia")
  }
''',
  initNs = { "vcard": VCARD }
)

# Imprime True si existe al menos uno, o False si no existe
for r in g.query(q):
  print(r)

False
